# CS781 Capstone - Phase 1: Legal LM Pretraining
## Architecture 2: LLaMA-Style Decoder (RoPE + RMSNorm + SwiGLU + GQA)

- **Model**: 12-layer decoder-only transformer (~62M params)
- **Key innovations**: Rotary Position Embeddings, RMSNorm, SwiGLU activation, Grouped-Query Attention
- **Tokenizer**: BPE trained on legal corpus (16K vocab, from A1)
- **Data**: All 4 legal sources (Supreme Court, High Courts, Acts, Tribunals)
- **Training**: AdamW, cosine LR decay, fp16, gradient accumulation

## 1. HuggingFace Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HuggingFace login successful.")

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import hashlib
import glob
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Model - LLaMA-style decoder (RoPE + RMSNorm + SwiGLU + GQA)
MODEL_DIM = 640
MODEL_LAYERS = 12
MODEL_HEADS = 10
MODEL_KV_HEADS = 2        # GQA: 5 query heads per KV head
MODEL_FF = 1706            # SwiGLU: floor(2/3 * 4 * 640)
MODEL_SEQ_LEN = 1024
MODEL_DROPOUT = 0.0

# Tokenizer
_tok_matches = glob.glob("/kaggle/input/**/legal_bpe_16384.json", recursive=True)
TOKENIZER_PATH = _tok_matches[0] if _tok_matches else "/kaggle/input/legal-bpe-tokenizer/legal_bpe_16384.json"
print(f"Found tokenizer at: {TOKENIZER_PATH}")
VOCAB_SIZE = 16384

# Training - 300M tokens (fits in one T4 session)
TRAIN_TOKENS = 300_000_000
VAL_TOKENS = 5_000_000
BATCH_SIZE = 4
GRAD_ACCUM = 8            # effective batch = 32 sequences = 32,768 tokens
PEAK_LR = 6e-4
MIN_LR = 6e-5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

# Generation
TARGET_WORDS = 128
MAX_GEN_TOKENS = 384
TEMPERATURE = 0.4
TOP_P = 0.92
TOP_K = 50
REPETITION_PENALTY = 1.15

# Output
OUTPUT_DIR = "/kaggle/working/capstone_arch2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")

## 4. BPE Tokenizer (from A1 - our own implementation)

In [ ]:
%%writefile bpe_tokenizer.py
"""
BPE Tokenizer - CS781 Assignment 1 implementation, optimized for Phase 1.

Speedups over original A1 code (same algorithm):
  1. fit() uses word frequencies + incremental pair-count updates
  2. encode() uses priority-based merging + word-level cache
  3. Checkpointing: saves progress every N merges so training survives Kaggle timeouts

Research-backed defaults:
  - vocab_size=16384: optimal for 100M-scale models (Zheng et al. 2024,
    "Scaling Laws with Vocabulary"). At 100M scale, 16K minimizes training
    cost; quality spread across 8K-256K is <2% BPB.
  - Corpus sample: 5-10M words is sufficient for merge stabilization.
"""

import json
import os
import time
from collections import defaultdict


class BPETokenizer:

    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos.extend(range(256))
        self.eow_id = len(self.itos)  # 260
        self.itos.append("</w>")
        self.merges = []
        self.vocab_size = len(self.itos)
        self._merge_rank = {}
        self._encode_cache = {}
        self._token_bytes = None
        self._ends_word = None

    # ------------------------------------------------------------------ #
    #  Training (with checkpointing)                                      #
    # ------------------------------------------------------------------ #

    def fit(self, corpus, checkpoint_dir=None, checkpoint_every=1000):
        ckpt_path = None
        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
            ckpt_path = os.path.join(checkpoint_dir, f"bpe_v{self.target_vocab_size}.ckpt.json")

        resumed_merge = 0
        word_data = None

        if ckpt_path and os.path.exists(ckpt_path):
            print(f"Found checkpoint at {ckpt_path}, resuming...")
            with open(ckpt_path, "r") as f:
                ckpt = json.load(f)
            self.merges = [((a, b), c) for (a, b), c in ckpt["merges"]]
            self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
            self.itos.extend(range(256))
            self.itos.append("</w>")
            for pair, new_id in self.merges:
                self.itos.append(pair)
            self.vocab_size = len(self.itos)
            resumed_merge = len(self.merges)
            word_data = []
            for tokens, freq in ckpt["words"]:
                word_data.append([tokens, freq])
            print(f"  Resumed after {resumed_merge} merges (vocab: {self.vocab_size})")

        if word_data is None:
            print("Counting word frequencies...")
            word_freq = {}
            for word in corpus.split():
                word_freq[word] = word_freq.get(word, 0) + 1
            print(f"  {len(word_freq):,} unique words")

            word_data = []
            for word, freq in word_freq.items():
                tokens = [b + 4 for b in word.encode("utf-8")]
                tokens.append(self.eow_id)
                word_data.append([tokens, freq])

        print("Building pair index...")
        pair_counts = defaultdict(int)
        pair_words = defaultdict(set)

        for idx, (tokens, freq) in enumerate(word_data):
            seen = defaultdict(int)
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                seen[pair] += 1
            for pair, cnt in seen.items():
                pair_counts[pair] += cnt * freq
                pair_words[pair].add(idx)

        num_merges_total = self.target_vocab_size - 261
        num_merges_left = num_merges_total - resumed_merge
        print(f"Learning {num_merges_left} merges (of {num_merges_total} total)...")
        t0 = time.time()

        for merge_i in range(num_merges_left):
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            if pair_counts[best_pair] < 2:
                break

            new_id = len(self.itos)
            bp_a, bp_b = best_pair

            affected = list(pair_words.get(best_pair, set()))

            for idx in affected:
                tokens, freq = word_data[idx]

                old_pairs = defaultdict(int)
                for i in range(len(tokens) - 1):
                    old_pairs[(tokens[i], tokens[i + 1])] += 1
                for p, cnt in old_pairs.items():
                    pair_counts[p] -= cnt * freq
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_words[p].discard(idx)
                    if not pair_words[p]:
                        pair_words.pop(p, None)

                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == bp_a and tokens[i + 1] == bp_b:
                        new_tokens.append(new_id)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                word_data[idx][0] = new_tokens

                new_pairs = defaultdict(int)
                for i in range(len(new_tokens) - 1):
                    new_pairs[(new_tokens[i], new_tokens[i + 1])] += 1
                for p, cnt in new_pairs.items():
                    pair_counts[p] = pair_counts.get(p, 0) + cnt * freq
                    pair_words[p].add(idx)

            self.merges.append((best_pair, new_id))
            self.itos.append(best_pair)
            self.vocab_size = len(self.itos)

            done = resumed_merge + merge_i + 1
            if done % 1000 == 0:
                elapsed = time.time() - t0
                rate = (merge_i + 1) / elapsed
                eta = (num_merges_left - merge_i - 1) / rate if rate > 0 else 0
                print(f"  {done}/{num_merges_total} merges | "
                      f"{elapsed:.0f}s elapsed | "
                      f"~{eta / 60:.1f} min remaining")

            if ckpt_path and (merge_i + 1) % checkpoint_every == 0:
                self._save_checkpoint(ckpt_path, word_data)

        if ckpt_path:
            self._save_checkpoint(ckpt_path, word_data)

        self._rebuild_caches()
        total_time = time.time() - t0
        print(f"Done. Final vocab: {self.vocab_size} | Time: {total_time / 60:.1f} min")
        return self

    def _save_checkpoint(self, path, word_data):
        ckpt = {
            "merges": [[[a, b], c] for (a, b), c in self.merges],
            "words": [[tokens, freq] for tokens, freq in word_data],
        }
        with open(path, "w") as f:
            json.dump(ckpt, f)

    # ------------------------------------------------------------------ #
    #  Encoding                                                           #
    # ------------------------------------------------------------------ #

    def _rebuild_caches(self):
        self._merge_rank = {}
        for rank, ((a, b), new_id) in enumerate(self.merges):
            self._merge_rank[(a, b)] = (rank, new_id)
        self._encode_cache = {}

        self._token_bytes = {x + 4: bytes([x]) for x in range(256)}
        self._token_bytes[self.eow_id] = b""
        self._ends_word = {i + 4: False for i in range(256)}
        self._ends_word[self.eow_id] = True
        for (a, b), c in self.merges:
            self._token_bytes[c] = self._token_bytes[a] + self._token_bytes[b]
            self._ends_word[c] = self._ends_word[b]

    def _encode_word(self, word):
        tokens = [b + 4 for b in word.encode("utf-8")]
        tokens.append(self.eow_id)

        while len(tokens) >= 2:
            best_rank = float("inf")
            best_idx = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                r = self._merge_rank.get(pair)
                if r is not None and r[0] < best_rank:
                    best_rank = r[0]
                    best_idx = i

            if best_idx == -1:
                break

            _, new_id = self._merge_rank[(tokens[best_idx], tokens[best_idx + 1])]
            tokens = tokens[:best_idx] + [new_id] + tokens[best_idx + 2:]

        return tokens

    def encode(self, text):
        if not self._merge_rank and self.merges:
            self._rebuild_caches()

        result = []
        for word in text.split():
            cached = self._encode_cache.get(word)
            if cached is not None:
                result.extend(cached)
            else:
                tokens = self._encode_word(word)
                self._encode_cache[word] = tokens
                result.extend(tokens)
        return result

    # ------------------------------------------------------------------ #
    #  Decoding                                                           #
    # ------------------------------------------------------------------ #

    def decode(self, tokens):
        if self._token_bytes is None:
            self._rebuild_caches()

        parts = []
        for i, token in enumerate(tokens):
            b = self._token_bytes.get(token)
            if b is not None:
                parts.append(b)
                if self._ends_word.get(token, False) and i < len(tokens) - 1:
                    parts.append(b" ")

        return b"".join(parts).decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ #
    #  Save / Load                                                        #
    # ------------------------------------------------------------------ #

    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "target_vocab_size": self.target_vocab_size,
            "merges": [[[a, b], c] for (a, b), c in self.merges],
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"Tokenizer saved to {path} ({self.vocab_size} tokens)")

    @classmethod
    def load(cls, path):
        with open(path, "r") as f:
            data = json.load(f)
        tok = cls(data["target_vocab_size"])
        tok.merges = [((a, b), c) for (a, b), c in data["merges"]]
        for pair, new_id in tok.merges:
            tok.itos.append(pair)
        tok.vocab_size = len(tok.itos)
        tok._rebuild_caches()
        return tok

    @property
    def pad_token_id(self):
        return 0

    @property
    def unk_token_id(self):
        return 1

    @property
    def bos_token_id(self):
        return 2

    @property
    def eos_token_id(self):
        return 3

    def __len__(self):
        return self.vocab_size

## 5. Load Trained Tokenizer

In [ ]:
from bpe_tokenizer import BPETokenizer

tokenizer = BPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded tokenizer: vocab_size={tokenizer.vocab_size}")

# sanity check
test_text = "The Supreme Court of India hereby declares"
enc = tokenizer.encode(test_text)
dec = tokenizer.decode(enc)
print(f"Encode: {test_text!r} -> {enc[:15]}...")
print(f"Decode: {dec!r}")
print(f"Tokens per word: {len(enc)/len(test_text.split()):.2f}")

## 6. Data Pipeline - Load, Clean, Tokenize (streaming)

Processes each source independently to stay within Kaggle's 13 GB RAM:
1. **Stream** JSON records with `ijson` (never loads full file into RAM)
2. **Clean** - dedup by MD5 of first 500 chars + remove docs < 50 words
3. **Tokenize** - each source gets an equal share of the token budget
4. Uses `array.array('H')` (2 bytes/token) instead of Python list (28 bytes/token)

In [ ]:
try:
    import ijson
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'ijson', '-q'], check=True)
    import ijson

import array
import gc
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

train_tokens = array.array('H')
val_tokens = array.array('H')

random.seed(SEED)
tokens_per_source = TRAIN_TOKENS // len(TRAIN_FILES)
val_per_source = VAL_TOKENS // len(TRAIN_FILES)

for train_file in TRAIN_FILES:
    source_name = train_file.split("/")[-1].replace("train_", "").replace(".json", "")
    print(f"\n{'─'*50}")
    print(f"Source: {source_name}")
    print(f"{'─'*50}")

    file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=train_file,
        repo_type="dataset", token=HF_TOKEN,
    )

    source_train_limit = len(train_tokens) + tokens_per_source
    source_val_limit = len(val_tokens) + val_per_source
    seen_hashes = set()
    doc_count = 0
    kept_count = 0

    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            doc_count += 1
            text = record["text"]

            h = hashlib.md5(text[:500].encode()).hexdigest()
            if h in seen_hashes:
                continue
            seen_hashes.add(h)

            if len(text.split()) < 50:
                continue

            kept_count += 1
            toks = tokenizer.encode(text)

            if random.random() < 0.05:
                if len(val_tokens) < source_val_limit:
                    val_tokens.extend(toks[:source_val_limit - len(val_tokens)])
            else:
                if len(train_tokens) < source_train_limit:
                    train_tokens.extend(toks[:source_train_limit - len(train_tokens)])

            if len(train_tokens) >= source_train_limit and len(val_tokens) >= source_val_limit:
                break

            if doc_count % 10000 == 0:
                print(f"  ... {doc_count:,} docs | {kept_count:,} kept | train={len(train_tokens):,} val={len(val_tokens):,}")

    del seen_hashes
    gc.collect()
    print(f"  Done: {doc_count:,} scanned, {kept_count:,} kept")
    print(f"  Tokens - train: {len(train_tokens):,} | val: {len(val_tokens):,}")

print(f"\n{'='*60}")
print(f"Final train tokens: {len(train_tokens):,}")
print(f"Final val tokens:   {len(val_tokens):,}")
print(f"Memory: {len(train_tokens)*2/1e9:.1f} GB (train) + {len(val_tokens)*2/1e9:.1f} GB (val)")

## 8. Token Block Dataset

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

# free tokenizer encode cache - it accumulated 100K+ entries during tokenization
tokenizer._encode_cache = {}
gc.collect()

class TokenBlockDataset(Dataset):
    def __init__(self, tokens, block_size):
        total = (len(tokens) // block_size) * block_size
        # numpy.frombuffer gives a zero-copy view of the array.array buffer,
        # avoiding the 20 GB Python-object intermediate that torch.ShortTensor() creates
        arr_np = np.frombuffer(tokens, dtype=np.uint16)[:total]
        self.data = torch.from_numpy(arr_np.astype(np.int16)).view(-1, block_size)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx].long()
        return x[:-1], x[1:]

train_dataset = TokenBlockDataset(train_tokens, MODEL_SEQ_LEN + 1)
val_dataset = TokenBlockDataset(val_tokens, MODEL_SEQ_LEN + 1)

del train_tokens, val_tokens
gc.collect()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training sequences:   {len(train_dataset):,}")
print(f"Validation sequences: {len(val_dataset):,}")

## 9. Model Architecture - GQA Attention with RoPE

In [ ]:
%%writefile attention.py
"""Grouped-Query Attention with Rotary Position Embeddings (RoPE)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    half = d_head // 2
    freqs = 1.0 / (base ** (torch.arange(0, half).float() / half))
    t = torch.arange(max_seq_len).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    T = x.shape[2]
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class CausalSelfAttention(nn.Module):

    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        cos, sin = build_rope_cache(max_seq_len, self.d_head)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out

## 10. Model Architecture - LLaMA-Style Decoder (RMSNorm + SwiGLU)

In [ ]:
%%writefile model_arch.py
"""LLaMA-style Decoder LM: RMSNorm, SwiGLU FFN, RoPE (in attention), GQA."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 12
    n_heads: int = 10
    n_kv_heads: int = 2
    d_ff: int = 1706
    max_seq_len: int = 1024
    dropout: float = 0.0


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads,
                                        cfg.max_seq_len, cfg.dropout)
        self.ln2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

## 11. Initialize Model with Proper Weight Init

In [ ]:
import torch.nn as nn
from model_arch import LMConfig, DecoderLM, RMSNorm

cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=MODEL_HEADS,
    n_kv_heads=MODEL_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MODEL_SEQ_LEN,
    dropout=MODEL_DROPOUT,
)

model = DecoderLM(cfg).to(device)

def init_weights(module):
    if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, RMSNorm):
        torch.nn.init.ones_(module.weight)

model.apply(init_weights)

residual_std = 0.02 / math.sqrt(2 * MODEL_LAYERS)
for block in model.blocks:
    torch.nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=residual_std)
    torch.nn.init.normal_(block.ffn.down_proj.weight, mean=0.0, std=residual_std)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params/1e6:.1f}M)")

## 12. Optimizer with Proper Weight Decay Groups

In [ ]:
# weight decay only on 2D params (Linear weights), NOT on biases + LayerNorm
decay_params = []
no_decay_params = []

for name, param in model.named_parameters():
    if param.dim() >= 2:
        decay_params.append(param)
    else:
        no_decay_params.append(param)

print(f"Decay params:    {sum(p.numel() for p in decay_params):,}")
print(f"No-decay params: {sum(p.numel() for p in no_decay_params):,}")

optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8)

## 13. Learning Rate Schedule (cosine with warmup)

In [ ]:
total_steps = len(train_loader) // GRAD_ACCUM
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {WARMUP_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM} sequences = {BATCH_SIZE * GRAD_ACCUM * MODEL_SEQ_LEN:,} tokens")

def get_lr(step):
    # linear warmup
    if step < WARMUP_STEPS:
        return PEAK_LR * step / WARMUP_STEPS
    # cosine decay to MIN_LR
    decay_ratio = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (PEAK_LR - MIN_LR)

## 14. Training Loop

In [ ]:
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

CKPT_SAVE_PATH = os.path.join(OUTPUT_DIR, "checkpoint.pt")
_ckpt_inputs = glob.glob("/kaggle/input/**/checkpoint.pt", recursive=True)
CKPT_LOAD_PATH = _ckpt_inputs[0] if _ckpt_inputs else CKPT_SAVE_PATH
CKPT_EVERY = 1000

scaler = GradScaler()
loss_history = []
val_losses = []

# resume from checkpoint if it exists
start_step = 0
if os.path.exists(CKPT_LOAD_PATH):
    print(f"Resuming from checkpoint: {CKPT_LOAD_PATH}")
    ckpt = torch.load(CKPT_LOAD_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    start_step = ckpt["step"]
    loss_history = ckpt.get("loss_history", [])
    val_losses = ckpt.get("val_losses", [])
    print(f"  Resumed at optimizer step {start_step}")

model.train()
step = 0
accum_loss = 0.0
t0 = time.time()

for context, target in train_loader:
    if step < start_step * GRAD_ACCUM:
        step += 1
        continue

    context, target = context.to(device), target.to(device)

    lr = get_lr(step // GRAD_ACCUM)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    with autocast():
        logits = model(context)
        B, L, V = logits.shape
        loss = F.cross_entropy(logits.view(B*L, V), target.view(B*L))
        loss = loss / GRAD_ACCUM

    scaler.scale(loss).backward()
    accum_loss += loss.item()

    if (step + 1) % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        actual_step = step // GRAD_ACCUM

        if actual_step % 100 == 0:
            elapsed = time.time() - t0
            tokens_seen = (actual_step + 1) * BATCH_SIZE * GRAD_ACCUM * MODEL_SEQ_LEN
            print(f"Step {actual_step:>6d} | loss={accum_loss:.4f} | lr={lr:.2e} | {elapsed:.0f}s | {tokens_seen/1e6:.1f}M tokens")
            loss_history.append({"step": actual_step, "loss": accum_loss, "tokens": tokens_seen})

        accum_loss = 0.0

        if actual_step % CKPT_EVERY == 0 and actual_step > 0:
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "step": actual_step,
                "loss_history": loss_history,
                "val_losses": val_losses,
            }, CKPT_SAVE_PATH)
            print(f"  >> Checkpoint saved at step {actual_step}")

        if actual_step % CKPT_EVERY == 0 and actual_step > 0:
            model.eval()
            val_loss_sum = 0.0
            val_count = 0
            with torch.no_grad():
                for vx, vy in val_loader:
                    vx, vy = vx.to(device), vy.to(device)
                    with autocast():
                        vlogits = model(vx)
                        vB, vL, vV = vlogits.shape
                        vloss = F.cross_entropy(vlogits.view(vB*vL, vV), vy.view(vB*vL))
                    val_loss_sum += vloss.item()
                    val_count += 1
            avg_val = val_loss_sum / val_count
            val_ppl = math.exp(min(avg_val, 20))
            val_losses.append({"step": actual_step, "val_loss": avg_val, "val_ppl": val_ppl})
            print(f"  >> Val loss={avg_val:.4f} | Val PPL={val_ppl:.1f}")
            model.train()

    step += 1

# final validation
model.eval()
val_loss_sum = 0.0
val_count = 0
with torch.no_grad():
    for vx, vy in val_loader:
        vx, vy = vx.to(device), vy.to(device)
        with autocast():
            vlogits = model(vx)
            vB, vL, vV = vlogits.shape
            vloss = F.cross_entropy(vlogits.view(vB*vL, vV), vy.view(vB*vL))
        val_loss_sum += vloss.item()
        val_count += 1

final_val_loss = val_loss_sum / val_count
final_ppl = math.exp(min(final_val_loss, 20))
print(f"\nFinal val loss: {final_val_loss:.4f} | Final PPL: {final_ppl:.1f}")
print(f"Total training time: {(time.time() - t0) / 60:.1f} min")

## 15. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# training loss
steps_plot = [h["step"] for h in loss_history]
losses_plot = [h["loss"] for h in loss_history]
ax1.plot(steps_plot, losses_plot)
ax1.set_xlabel("Step")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss Curve")
ax1.grid(True, alpha=0.3)

# validation perplexity
if val_losses:
    val_steps = [v["step"] for v in val_losses]
    val_ppls = [v["val_ppl"] for v in val_losses]
    ax2.plot(val_steps, val_ppls, marker='o', color='orange')
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Validation Perplexity")
    ax2.set_title("Validation Perplexity")
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150)
plt.show()
print("Loss curves saved.")

## 16. Generation Function (top-p nucleus sampling)

In [ ]:
def generate(model, tokenizer, prompt, max_tokens=384, target_words=128,
             temperature=0.4, top_p=0.92, top_k=50, repetition_penalty=1.15):
    model.eval()
    tokens = tokenizer.encode(prompt)
    generated = []

    with torch.no_grad():
        for _ in range(max_tokens):
            context = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
            # truncate to max seq len (sliding window)
            context = context[:, -cfg.max_seq_len:]

            with autocast():
                logits = model(context)
            logits = logits[0, -1].float()  # (vocab_size,)

            # repetition penalty: penalize tokens already generated
            for token_id in set(generated[-100:]):  # look back 100 tokens
                logits[token_id] /= repetition_penalty

            # temperature scaling
            logits = logits / temperature

            # top-k filtering
            if top_k > 0:
                top_k_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_k_vals[-1]] = float('-inf')

            # top-p (nucleus) filtering
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
            sorted_indices_to_remove[0] = False
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[indices_to_remove] = float('-inf')

            # sample from the filtered distribution
            probs = torch.softmax(logits, dim=-1)
            token_id = torch.multinomial(probs, num_samples=1).item()

            tokens.append(token_id)
            generated.append(token_id)

            # check word count
            decoded_so_far = tokenizer.decode(generated)
            if len(decoded_so_far.split()) >= target_words:
                break

    # decode all generated tokens at once and trim to target_words
    continuation = tokenizer.decode(generated)
    words = continuation.split()[:target_words]
    return " ".join(words)

## 17. Load Test Data

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Number of test examples: {len(test_data)}")

# show a sample prompt
sample_id = list(test_data.keys())[0]
print(f"\nSample prompt (id={sample_id}):")
print(test_data[sample_id]["text"][:300] + "...")

## 18. Generate Predictions

In [ ]:
predictions = {}
items = list(test_data.items())
total_examples = len(items)
print(f"Generating {total_examples} predictions...")

t0 = time.time()
for i, (example_id, record) in enumerate(items):
    prompt = record["text"]
    prediction = generate(
        model, tokenizer, prompt,
        max_tokens=MAX_GEN_TOKENS,
        target_words=TARGET_WORDS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=REPETITION_PENALTY,
    )
    predictions[str(example_id)] = {"text": prediction}

    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{total_examples} ({100*(i+1)/total_examples:.1f}%) | {elapsed:.0f}s")

print(f"\nGeneration complete in {(time.time()-t0)/60:.1f} min")

## 19. Sample Predictions

In [ ]:
for i, (eid, pred) in enumerate(list(predictions.items())[:5]):
    prompt = test_data[eid]["text"]
    print(f"\n{'='*60}")
    print(f"Example {eid}")
    print(f"PROMPT: {prompt[:200]}...")
    print(f"GENERATED ({len(pred['text'].split())} words): {pred['text'][:300]}...")

## 20. Save Predictions & Create Submission

In [ ]:
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"prediction.json saved: {PREDICTION_PATH}")
print(f"submission.zip created: {SUBMISSION_ZIP}")
print(f"\nPrediction count: {len(predictions)}")
avg_words = sum(len(p['text'].split()) for p in predictions.values()) / len(predictions)
print(f"Avg generated words: {avg_words:.1f}")

## 21. Save Model & Metrics

In [ ]:
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model.pt"))

metrics = {
    "architecture": "llama_style_rope_rmsnorm_swiglu_gqa",
    "params": n_params,
    "d_model": MODEL_DIM,
    "n_layers": MODEL_LAYERS,
    "n_heads": MODEL_HEADS,
    "n_kv_heads": MODEL_KV_HEADS,
    "d_ff": MODEL_FF,
    "seq_len": MODEL_SEQ_LEN,
    "vocab_size": VOCAB_SIZE,
    "train_tokens": len(train_dataset) * (MODEL_SEQ_LEN + 1),
    "final_val_loss": final_val_loss,
    "final_ppl": final_ppl,
    "loss_history": loss_history,
    "val_losses": val_losses,
    "generation_config": {
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "repetition_penalty": REPETITION_PENALTY,
    },
}

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("Model and metrics saved.")
print(f"\n{'='*60}")
print(f"SUBMISSION READY: {SUBMISSION_ZIP}")
print(f"Model: {n_params/1e6:.1f}M params | Val PPL: {final_ppl:.1f}")
print(f"{'='*60}")